# **End-to-End RAG Project using OpenAI SDK**

This notebook teaches you how to build a **simple Retrieval-Augmented Generation (RAG)** chatbot using:

- **OpenAI Responses API** for chat
- **Vector Stores + file_search** for retrieval over your documents
- **Conversations** for multi-turn chat memory

You will learn how to:
1. Upload documents and index them into a Vector Store
2. Ask questions that are answered using your documents
3. Add new documents later and keep chatting in the same conversation
4. Enable a “Strict Librarian” mode to refuse answers not found in your docs
5. Clean up cloud resources (vector stores and files) when you are done

Prerequisites:
- Intermediate comfort with Python functions and dictionaries
- Your OpenAI API key available as an environment variable: `OPENAI_API_KEY`



In [ ]:
# Install the latest version of the OpenAI SDK
!pip install -U openai

---

### **3. Authentication & Configuration**

To use the API, we need a "key" (like a password). We load this securely from Google Colab's secrets manager.

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

# 1. Get the API Key
# Make sure you've added 'OPENAI_API_KEY' in the Colab secrets tab (Key icon on left)
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. Initialize the Client
# This 'client' object is our main gateway to all OpenAI functions.
client = OpenAI()

print("✅ Client initialized successfully.")

✅ Client initialized successfully.


---

### **4. Creating the "Private" Knowledge Base**

For RAG to work, we need data. In a real company, this would be PDFs or Spreadsheets. For this lesson, we will create a dummy text file representing "Secret Project Omega."

In [ ]:
# 1. Create a dummy text file named 'project_omega.txt'
# We are writing secret facts that the public ChatGPT definitely doesn't know.
secret_content = """
Project Omega Confidential Briefing:
1. The launch date is set for November 15th, 2026.
2. The lead engineer is Dr. Aris Thorne.
3. The budget has been approved at $50 million.
4. The secret passcode for the lab is 'Blue-Horizon'.
"""

with open("project_omega.txt", "w") as f:
    f.write(secret_content)

print("✅ Dummy file created: project_omega.txt")

# 2. Create a Vector Store (our hosted knowledge base) and upload the file into it.
# This is required for the Responses API `file_search` tool.
# Docs: Vector stores + upload_and_poll helper.
vector_store = client.vector_stores.create(name="Project Omega KB")

with open("project_omega.txt", "rb") as f:
    vs_file = client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=f,
    )

print(f"✅ Vector store created: {vector_store.id}")
print(f"✅ File indexed in vector store. Status: {vs_file.status}")

✅ Dummy file created: project_omega.txt
✅ Vector store created: vs_69a853c269f881919f1f85b575102df4
✅ File indexed in vector store. Status: completed


---

### **5. Setting up "Memory" (The Conversation)**

In the old days, we had to send the chat history back and forth manually. Now, we use the **Conversations API**. This creates a persistent "session" on the server that remembers what we said.

In [ ]:
# Create a Conversation object
# This will hold the history of our chat so we can ask follow-up questions.
conversation = client.conversations.create()

print(f"✅ Conversation Started. ID: {conversation.id}")

✅ Conversation Started. ID: conv_69a853ca10948197b4bbde251c520eb30f571d4378cef8e9


---

### **6. The Chat Function (The Core Logic)**

This is the most important part of the code. We will wrap the complexity of the **Responses API** into a simple function called `ask_bot`.

**What does this function do?**

1. Takes your question.
2. Sends it to OpenAI with the **file_search** tool configured to use our **Vector Store** (our file).
3. Receives the answer.
4. **Crucial Step:** It looks for "citations" (footnotes) to prove where it got the info.

In [ ]:
def ask_bot(user_question):
    """
    Sends a user question to the OpenAI Responses API using the specified vector store
    for retrieval-augmented generation (RAG).

    Args:
        user_question (str): The question to ask the bot.
    """
    print(f"User: {user_question}")
    print("Thinking and searching files...")

    # 1. Call the OpenAI Responses API
    # The 'responses' endpoint is designed for multi-turn chat with tool capabilities.
    # We pass 'conversation_id' so the model maintains context of previous messages.
    response = client.responses.create(
        model="gpt-5-mini",
        conversation=conversation.id,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": user_question}
                ],
            }
        ],
        # 2. Configure Tools
        # 'file_search' allows the model to query our uploaded documents.
        # We specify 'vector_store_ids' to restrict the search to our 'Project Omega' store.
        tools=[
            {
                "type": "file_search",
                "vector_store_ids": [vector_store.id],
                "max_num_results": 5,  # Retrieve up to 5 relevant chunks
            }
        ],
    )

    # 3. Extract the final text answer
    # We use getattr to safely access attributes in case the response structure varies
    answer = getattr(response, "output_text", "") or ""

    # 4. Extract Citations
    # The API returns annotations (like footnotes) indicating which files were used.
    citations = []
    for item in getattr(response, "output", []) or []:
        # Look for message items
        if getattr(item, "type", None) != "message":
            continue
        for content in getattr(item, "content", []) or []:
            # Look for text output content
            if getattr(content, "type", None) != "output_text":
                continue
            # Check annotations for file citations
            for ann in getattr(content, "annotations", []) or []:
                if getattr(ann, "type", None) == "file_citation":
                    fn = getattr(ann, "filename", None)
                    # Collect unique filenames
                    if fn and fn not in citations:
                        citations.append(fn)

    # 5. Print the Result
    print()
    print("AI:", answer.strip())

    # 6. Print Sources (if any were found)
    if citations:
        print()
        print("--- Sources ---")
        for i, fn in enumerate(citations, start=1):
            print(f"[{i}] {fn}")

    print("-" * 50)

---

### **7. Testing the System**

Now, let's see if our RAG system works. We will ask questions that the AI could **only** know if it successfully read our file.

**Test 1: Specific Fact Retrieval**
The model should find the specific name "Dr. Aris Thorne".

In [ ]:
ask_bot("Who is the lead engineer for the project?")

User: Who is the lead engineer for the project?
Thinking and searching files...

AI: The lead engineer for Project Omega is Dr. Aris Thorne .

Do you mean a different project or would you like contact/details for Dr. Thorne?

--- Sources ---
[1] project_omega.txt
--------------------------------------------------


**Test 2: Contextual Follow-up (Memory)**
We won't mention the project name or the engineer's name. We'll just ask "What is the budget?". The `conversation` ID we passed earlier ensures the AI knows what we are talking about.

In [ ]:
ask_bot("And what is the approved budget?")

User: And what is the approved budget?
Thinking and searching files...

AI: The approved budget is $50 million, per the Project Omega confidential briefing .

Would you like a budget breakdown or any other details from the briefing?

--- Sources ---
[1] project_omega.txt
--------------------------------------------------


**Test 3: Security Check**
Let's ask for the passcode.

In [ ]:
ask_bot("What is the passcode for the lab?")

User: What is the passcode for the lab?
Thinking and searching files...

AI: The secret passcode for the lab is "Blue-Horizon", per the Project Omega confidential briefing .

This is sensitive information — would you like me to store it securely or share it with anyone else?

--- Sources ---
[1] project_omega.txt
--------------------------------------------------


---

### **Summary of Concepts Used**

| Concept | Code Object | Analogy |
| --- | --- | --- |
| **Brain** | `gpt-5-mini` | The smart student taking the test. |
| **Knowledge Base** | `Vector Store` | The textbook the student is allowed to open. |
| **Memory** | `Conversation` | The transcript of what has been said so far. |
| **Retriever** | `file_search` | The student's finger pointing to the specific page in the book. |

## A) Add Files Later to the Same Vector Store (and Keep Chatting)

In real projects, your document set evolves over time. This section shows how to:

1. Add `abs.pdf` from your working directory into the *same* Vector Store
2. Continue the *same* conversation (no need to start over)
3. Add a PDF from a URL (GitHub raw link, Google Drive link, and any direct PDF URL), and then continue the conversation

We will implement this using small helper functions so the flow stays clear.

In [ ]:
vector_store.id

NameError: name 'vector_store' is not defined

In [ ]:
conversation.id

'conv_696d16a9d6808190903920962835ddb80cd6c5217c0397bf'

In [ ]:
# Assumes you already created these earlier in the notebook:
# - vector_store.id
# - conversation.id

In [ ]:
def add_local_pdf_to_vector_store(vector_store_id: str, pdf_path: str) -> str:
    """Add a LOCAL PDF to an existing Vector Store.

    Why this function exists:
    - You may want to add more documents over time (e.g., another PDF next week).
    - Once the PDF is in the vector store, file_search can retrieve relevant chunks.

    Returns:
        file_id (str): The uploaded file's ID (useful later for cleanup).

    Notes:
        We use `vector_stores.files.upload_and_poll(...)` because indexing is asynchronous.
        This helper waits until the file is indexed and ready for search.
    """

    # Upload the file and wait until indexing completes
    vs_file = client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store_id,
        file=open(pdf_path, "rb"),
    )

    # vs_file contains both:
    # - vs_file.id      : the vector-store-file association record
    # - vs_file.vector_store_id : contians the vector store id
    return vs_file.id

In [ ]:
vs_file

VectorStoreFile(id='file-FdVBRC3yBr7V8LoJ4JX2rC', created_at=1768756832, last_error=None, object='vector_store.file', status='completed', usage_bytes=1254, vector_store_id='vs_696d165da4588191a7075deda67e89e8', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [ ]:
# (Replace the filename below with your PDF filename)
file_id_abs = add_local_pdf_to_vector_store(vector_store.id, "Advanced RAG Strategies in 2026.pdf")
print("Uploaded PDF file_id:", file_id_abs)

Uploaded PDF file_id: file-CZcN81Ph3wrxqbfFnz2c4T


### 2) Continue the Conversation Using the Same Vector Store

Because we keep the same `conversation.id`, the model can maintain chat context across turns.

Because we keep the same `vector_store.id`, it can retrieve from *all* files currently indexed in that store.

In [ ]:
def ask_with_rag(conversation_id: str, vector_store_id: str, question: str, model: str = "gpt-5-mini") -> str:
    response = client.responses.create(
        model=model,
        conversation=conversation_id,
        input=[{"role": "user", "content": question}],
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
            "max_num_results": 20,
        }],
    )
    return response.output_text

In [ ]:
print(ask_with_rag(conversation.id, vector_store.id, "List the advsnced RAG strategies"))

I checked your uploaded files; they only contain the Project Omega briefing and don’t include RAG strategies . Below are advanced Retrieval‑Augmented Generation (RAG) strategies you can apply:

- Hybrid retrieval (dense + sparse): combine BM25/TF‑IDF with vector search to cover exact matches and semantic recall.
- Multi‑stage retrieval (coarse → fine): use a fast retriever to get candidates and a cross‑encoder re‑ranker to refine the top results.
- Fusion‑in‑Decoder (FiD): retrieve multiple passages and let the decoder attend to them jointly for better multi‑document fusion.
- Rerank‑then‑generate: apply a strong cross‑encoder or learned reranker, then feed top‑k passages to the generator.
- Query reformulation / expansion: rewrite or expand queries (LLM‑assisted or learned) to increase recall and surface varied phrasings.
- Iterative / multi‑hop retrieval: perform stepwise retrieval where each generation step issues new queries to fetch deeper context.
- Retrieval‑aware prompting: inc

## B) Enhancement 1 — Strict Librarian Mode

Many RAG applications must **refuse** to answer when the documents do not contain the information.

We enforce this in two ways:
1. **System-style instructions**: Tell the model to answer only from documents
2. **Tool enforcement**: Set `tool_choice="required"` so retrieval is always used

If the answer is not found in the documents, the bot returns a standard refusal message.

In [ ]:
vector_store.id

'vs_696d165da4588191a7075deda67e89e8'

In [ ]:
STRICT_LIBRARIAN_INSTRUCTIONS = '''
You are a strict documentation assistant.

Rules:
1) Answer ONLY using information found in the provided internal documents via file_search.
2) If you cannot find the answer in those documents, reply exactly:
   "I could not find that information in the internal documents."
3) Do not use outside knowledge. Do not guess.
4) If the user asks something unrelated to the documents, follow rule #2.
'''


def ask_bot_strict(conversation_id: str, vector_store_id: str, question: str, model: str = "gpt-5-mini") -> str:
    response = client.responses.create(
        model=model,
        conversation=conversation_id,
        input=[{"role": "user", "content": question}],

        # "Strict Librarian" behavior
        instructions=STRICT_LIBRARIAN_INSTRUCTIONS,

        # Force retrieval (prevents answering from general knowledge)
        tool_choice="required",
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
            "max_num_results": 20,
        }],
    )

    return response.output_text


In [ ]:
# --- Tests ---
print("IN-DOC:", ask_bot_strict(conversation.id, vector_store.id, "What is the budget for Project Omega?"))

IN-DOC: The approved budget for Project Omega is $50 million, per the Project Omega confidential briefing .


In [ ]:
conversation.id, vector_store.id

('conv_696d16a9d6808190903920962835ddb80cd6c5217c0397bf',
 'vs_696d165da4588191a7075deda67e89e8')

In [ ]:
print("IN-DOC:", ask_bot_strict(conversation.id, vector_store.id, "What is RAG?"))

IN-DOC: I could not find that information in the internal documents.


In [ ]:
print("OUT-OF-DOC:", ask_bot_strict(conversation.id, vector_store.id, "What is the capital of France?"))

OUT-OF-DOC: I could not find that information in the internal documents.


## C) Enhancement 2 — Resource Cleanup (Good Cloud Hygiene)

When teaching or prototyping, it is easy to create many cloud resources.

Two cleanup levels are common:

1. **Minimal cleanup**: delete the vector store (removes the search index)
2. **Full cleanup**: delete the vector store **and** delete the uploaded files from OpenAI storage

Best practice: keep a list of uploaded `file_id`s so you can delete them later.

In [ ]:
file_id_abs

'file-CZcN81Ph3wrxqbfFnz2c4T'

In [ ]:
def cleanup_vector_store_only(vector_store_id: str):
    """
    Minimal cleanup: delete the vector store (search index).
    This does NOT delete the underlying file objects.
    """
    client.vector_stores.delete(vector_store_id)
    print(f"Deleted vector store: {vector_store_id}")


def cleanup_everything(vector_store_id: str, file_ids: list[str]):
    """
    Full cleanup: delete the vector store AND delete the file objects.

    Parameters:
        vector_store_id: Vector store to delete
        file_ids: list of uploaded file IDs to delete
    """

    # 1) Delete the vector store (index)
    client.vector_stores.delete(vector_store_id)
    print(f"Deleted vector store: {vector_store_id}")

    # 2) Delete uploaded files from OpenAI storage
    for fid in file_ids:
        try:
            client.files.delete(fid)
            print(f"Deleted file: {fid}")
        except Exception as e:
            print(f"Could not delete file {fid}: {e}")


In [ ]:
# Keep track of uploaded file IDs as you add documents (so you can clean them up later)
uploaded_file_ids = []

# Example: track IDs whenever you upload
uploaded_file_ids.append(file_id_abs)

In [ ]:
uploaded_file_ids

['file-CZcN81Ph3wrxqbfFnz2c4T']

In [ ]:
# --- Usage examples (uncomment when you are done with the project) ---
# cleanup_vector_store_only(vector_store.id)
cleanup_everything(vector_store.id, uploaded_file_ids)

Deleted vector store: vs_696d165da4588191a7075deda67e89e8
Deleted file: file-CZcN81Ph3wrxqbfFnz2c4T
